In [1]:
import os
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
import torch
from huggingface_hub import hf_hub_download
import sys
sys.path.append(str(Path(os.getcwd()).parent))
from pipeline_flux import FluxPipeline
from transformer_flux import FluxTransformer2DModel
device_str = 'cuda'

In [ ]:
#original flux
# from diffusers import FluxPipeline
# bfl_repo="black-forest-labs/FLUX.1-dev"
# device = torch.device(device_str)
# dtype = torch.bfloat16
# pipe = FluxPipeline.from_pretrained(bfl_repo, torch_dtype=torch.bfloat16)
# pipe = pipe.to(device)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

In [2]:
bfl_repo="black-forest-labs/FLUX.1-dev"
device = torch.device(device_str)
dtype = torch.bfloat16
transformer = FluxTransformer2DModel.from_pretrained(bfl_repo, subfolder="transformer", torch_dtype=dtype)
pipe = FluxPipeline.from_pretrained(bfl_repo, transformer=transformer, torch_dtype=dtype)
pipe.scheduler.config.use_dynamic_shifting = False
pipe.scheduler.config.time_shift = 10
pipe = pipe.to(device)

/opt/anaconda3/envs/lwd/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Expected types for transformer: (<class 'diffusers.models.transformers.transformer_flux.FluxTransformer2DModel'>,), got <class 'transformer_flux.FluxTransformer2DModel'>.


In [3]:
#urae original
if not os.path.exists('../ckpt/urae_2k_adapter.safetensors'):
    hf_hub_download(repo_id="Huage001/URAE", filename='urae_2k_adapter.safetensors', local_dir='ckpt', local_dir_use_symlinks=False)
pipe.load_lora_weights("../ckpt/urae_2k_adapter.safetensors")
pipe = pipe.to(device)

No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


In [6]:
#our trained URAE
checkpoint_path = "/mnt/media/luigi/LWD Material/Checkpoints/URAE/2K/URAE_VAE_SE_WAV_ATT_LAION/checkpoint-2000/"
pipe.load_lora_weights(checkpoint_path)
pipe = pipe.to(device)

/opt/anaconda3/envs/lwd/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


In [8]:
#our trained URAE
checkpoint_path = "/mnt/media/luigi/LWD Material/Checkpoints/URAE/2K/URAE_original_trained_by_me/checkpoint-2000"
pipe.load_lora_weights(checkpoint_path)
pipe = pipe.to(device)

No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


In [12]:
#our URAE with wav att and without vae trained
checkpoint_path = "/mnt/media/luigi/LWD Material/Checkpoints/URAE/2K/l_003_URAE_VAE_NOOO_SE_WAV_ATT_LAION_2048/checkpoint-6000"
pipe.load_lora_weights(checkpoint_path)
pipe = pipe.to(device)

No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


In [14]:
prompt = "A serene woman in a flowing azure dress, gracefully perched on a sunlit cliff overlooking a tranquil sea, her hair gently tousled by the breeze. The scene is infused with a sense of peace, evoking a dreamlike atmosphere, reminiscent of Impressionist paintings."
height = 2048
width = 2048
image = pipe(
    prompt,
    height=height,
    width=width,
    guidance_scale=3.5,
    num_inference_steps=28,
    max_sequence_length=512,
    generator=torch.manual_seed(8888),
    ntk_factor=10,
    proportional_attention=True
).images[0]
image

  0%|          | 0/28 [00:00<?, ?it/s]

KeyboardInterrupt: 